In [17]:
"""
Wide vs Narrow Transformations — PySpark 4.0
============================================
Narrow transformation — each output partition depends on AT MOST ONE
  input partition.  Data never crosses executor boundaries.
  Examples: filter, select, withColumn, map, union, coalesce.

Wide transformation  — each output partition may depend on MANY input
  partitions.  Requires a shuffle: data is written to disk, redistributed
  across the network, and read back.
  Examples: groupBy, join (shuffle), repartition, distinct, sort.

Consequences
  Narrow  → pipelined into one stage, fused by whole-stage codegen,
            cheap fault recovery (one parent partition to recompute).
  Wide    → stage boundary, Exchange node in plan, disk I/O,
            expensive fault recovery (may need all parent partitions).
"""

import time

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [18]:
def section(title: str) -> None:
    bar = "=" * 70
    print(f"\n{bar}\n  {title}\n{bar}\n")


def timed(label: str, action):
    t0 = time.perf_counter()
    result = action()
    elapsed = time.perf_counter() - t0
    print(f"{label}: {elapsed:.3f}s")
    return result


def exchange_count(df) -> int:
    """Count Exchange nodes in the physical plan as a proxy for shuffle stages."""
    return df._jdf.queryExecution().executedPlan().toString().count("Exchange")


spark = (
    SparkSession.builder
    .appName("WideVsNarrow")
    .master("local[8]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.adaptive.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

In [19]:
# ═══════════════════════════════════════════════════════════════════════════ #
# 1. TAXONOMY — which operations are narrow and which are wide
#
#  The presence of an Exchange node in the physical plan is the definitive
#  indicator of a wide (shuffle) transformation.  Narrow operations produce
#  no Exchange — their output partitions derive from a single input partition.
# ═══════════════════════════════════════════════════════════════════════════ #
section("1. Taxonomy — narrow vs wide operations")
print("""
  Exchange node present  →  wide  (shuffle required)
  Exchange node absent   →  narrow (no data movement)
""")

base = (
    spark.range(100_000)
    .withColumn("dept",  (F.col("id") % 5).cast("int"))
    .withColumn("value", F.col("id").cast("double"))
)
base2 = (
    spark.range(100_000)
    .withColumn("dept", (F.col("id") % 5).cast("int"))
    .withColumn("value", (F.col("id") * 2).cast("double"))
)

narrow_ops = {
    "filter"      : base.filter(F.col("value") > 0),
    "select"      : base.select("id", "dept"),
    "withColumn"  : base.withColumn("doubled", F.col("value") * 2),
    "union"       : base.union(base2),
    "coalesce"    : base.coalesce(4),
}

wide_ops = {
    "groupBy"     : base.groupBy("dept").agg(F.sum("value")),
    "repartition" : base.repartition(4),
    "distinct"    : base.select("dept").distinct(),
    "sort"        : base.sort("value"),
    "join"        : base.join(base2, "dept"),
}

print(f"{'operation':<16} {'type':<8} {'Exchange nodes'}")
print("-" * 42)
for name, df in narrow_ops.items():
    print(f"{name:<16} {'narrow':<8} {exchange_count(df)}")
for name, df in wide_ops.items():
    print(f"{name:<16} {'wide':<8} {exchange_count(df)}")


  1. Taxonomy — narrow vs wide operations


  Exchange node present  →  wide  (shuffle required)
  Exchange node absent   →  narrow (no data movement)

operation        type     Exchange nodes
------------------------------------------
filter           narrow   0
select           narrow   0
withColumn       narrow   0
union            narrow   0
coalesce         narrow   0
groupBy          wide     1
repartition      wide     1
distinct         wide     1
sort             wide     1
join             wide     1


In [20]:
# ═══════════════════════════════════════════════════════════════════════════ #
# 2. WHOLE-STAGE CODEGEN FUSION
#
#  Spark's Tungsten engine compiles chains of narrow operators into a single
#  Java class per stage — "whole-stage codegen".  The * prefix in explain()
#  marks fused operators; [codegen id : N] shows which stage they belong to.
#
#  A wide transformation breaks the fusion: the Exchange is the boundary.
#  Operators before it compile into one codegen region; operators after
#  compile into a new one.
# ═══════════════════════════════════════════════════════════════════════════ #
section("2. Whole-stage codegen fusion")
print("""
  Look for:  * prefix on operators = fused into one compiled Java class.
             [codegen id : N] shows the stage/region boundary.
             All narrow operators share codegen id 1.
             Exchange resets the counter — operators after it get id 2.
""")

df = spark.range(1_000_000).withColumn("dept", (F.col("id") % 5).cast("int"))

narrow_chain = (
    df
    .filter(F.col("id") > 0)
    .withColumn("doubled", F.col("id") * 2)
    .withColumn("category", (F.col("id") % 3).cast("string"))
    .select("id", "dept", "doubled", "category")
)

print("--- narrow chain only (one codegen region) ---")
narrow_chain.explain("formatted")

wide_chain = narrow_chain.groupBy("dept").agg(F.sum("doubled").alias("total"))

print("--- narrow chain + groupBy (two codegen regions, Exchange between) ---")
wide_chain.explain("formatted")


  2. Whole-stage codegen fusion


  Look for:  * prefix on operators = fused into one compiled Java class.
             [codegen id : N] shows the stage/region boundary.
             All narrow operators share codegen id 1.
             Exchange resets the counter — operators after it get id 2.

--- narrow chain only (one codegen region) ---
== Physical Plan ==
* Project (3)
+- * Filter (2)
   +- * Range (1)


(1) Range [codegen id : 1]
Output [1]: [id#130L]
Arguments: Range (0, 1000000, step=1, splits=Some(8))

(2) Filter [codegen id : 1]
Input [1]: [id#130L]
Condition : (id#130L > 0)

(3) Project [codegen id : 1]
Output [4]: [id#130L, cast((id#130L % 5) as int) AS dept#131, (id#130L * 2) AS doubled#132L, cast((id#130L % 3) as string) AS category#133]
Input [1]: [id#130L]


--- narrow chain + groupBy (two codegen regions, Exchange between) ---
== Physical Plan ==
* HashAggregate (6)
+- Exchange (5)
   +- * HashAggregate (4)
      +- * Project (3)
         +- * Filter (2)
            

In [22]:
# ═══════════════════════════════════════════════════════════════════════════ #
# 3. RDD LINEAGE — toDebugString()
#
#  Every RDD records how it was derived from its parents.  Narrow transforms
#  produce simple one-to-one chains (OneToOneDependency).  Wide transforms
#  introduce a ShuffledRDD, which depends on ALL parent partitions.
#
#  Long lineage chains from narrow transforms are cheap — losing a partition
#  only requires recomputing that one partition's ancestry.
#  A ShuffledRDD boundary is expensive to recover — losing a reducer
#  partition may require rerunning all mapper tasks.
# ═══════════════════════════════════════════════════════════════════════════ #
section("3. RDD lineage — toDebugString()")
print("""
  toDebugString() prints the full RDD dependency graph.
  Indentation represents dependency depth.

  Look for:  narrow chain → flat indented MapPartitionsRDD chain.
             wide op      → ShuffledRDD appears, ancestry resets below it.
""")

narrow = (
    spark.range(10_000)
    .filter(F.col("id") > 0)
    .withColumn("v", F.col("id") * 2)
    .filter(F.col("v") < 15_000)
)

wide = narrow.groupBy(F.col("v") % 3).agg(F.count("*"))

print("--- narrow chain lineage ---")
print(narrow.rdd.toDebugString().decode())

print("--- wide (groupBy) lineage — ShuffledRDD appears ---")
print(wide.rdd.toDebugString().decode())


  3. RDD lineage — toDebugString()


  toDebugString() prints the full RDD dependency graph.
  Indentation represents dependency depth.

  Look for:  narrow chain → flat indented MapPartitionsRDD chain.
             wide op      → ShuffledRDD appears, ancestry resets below it.

--- narrow chain lineage ---
(8) MapPartitionsRDD[103] at javaToPython at NativeMethodAccessorImpl.java:0 []
 |  MapPartitionsRDD[102] at javaToPython at NativeMethodAccessorImpl.java:0 []
 |  SQLExecutionRDD[101] at javaToPython at NativeMethodAccessorImpl.java:0 []
 |  MapPartitionsRDD[100] at javaToPython at NativeMethodAccessorImpl.java:0 []
 |  MapPartitionsRDD[99] at javaToPython at NativeMethodAccessorImpl.java:0 []
 |  ParallelCollectionRDD[98] at javaToPython at NativeMethodAccessorImpl.java:0 []
--- wide (groupBy) lineage — ShuffledRDD appears ---
(8) MapPartitionsRDD[112] at javaToPython at NativeMethodAccessorImpl.java:0 []
 |  MapPartitionsRDD[111] at javaToPython at NativeMethodAccessorImpl.java:0

In [23]:
# ═══════════════════════════════════════════════════════════════════════════ #
# 4. CHECKPOINTING — TRUNCATING EXPENSIVE LINEAGE
#
#  After many wide transformations, the lineage graph grows deep and
#  recomputing a lost partition becomes very expensive — potentially
#  replaying multiple shuffle stages.
#
#  checkpoint() materialises the DataFrame to reliable storage and cuts
#  the lineage at that point.  Subsequent operations rebuild from the
#  checkpoint file, not from the original source.
#
#  Note: checkpoint() returns a NEW DataFrame — the original variable must
#  be reassigned to observe the truncated plan.
# ═══════════════════════════════════════════════════════════════════════════ #
section("4. Checkpointing — truncating expensive lineage")
print("""
  Build a deep lineage via multiple wide ops, then checkpoint.

  Look for:  explain() before checkpoint shows the full multi-stage plan
             with multiple Exchange nodes.
             After checkpoint the plan collapses to a single ExistingRDD
             scan — Spark has no knowledge of what came before the file.
""")

import tempfile, shutil
checkpoint_dir = tempfile.mkdtemp()
spark.sparkContext.setCheckpointDir(checkpoint_dir)

df = spark.range(10_000).withColumn("key", F.col("id") % 5)

step1 = df.groupBy("key").agg(F.sum("id").alias("s1"))
step2 = step1.withColumn("s2", F.col("s1") * 2).groupBy("s2").agg(F.count("*").alias("c"))
step3 = step2.repartition(4)

print("--- plan before checkpoint (three wide ops, two Exchange nodes) ---")
step3.explain("formatted")

# checkpoint(eager=True) is the default — materialises immediately.
# Must reassign: checkpoint() returns a new DataFrame backed by the file.
step3 = step3.checkpoint()

print("--- plan after checkpoint (single ExistingRDD scan, lineage gone) ---")
step3.explain("formatted")

shutil.rmtree(checkpoint_dir, ignore_errors=True)


  4. Checkpointing — truncating expensive lineage


  Build a deep lineage via multiple wide ops, then checkpoint.

  Look for:  explain() before checkpoint shows the full multi-stage plan
             with multiple Exchange nodes.
             After checkpoint the plan collapses to a single ExistingRDD
             scan — Spark has no knowledge of what came before the file.

--- plan before checkpoint (three wide ops, two Exchange nodes) ---
== Physical Plan ==
Exchange (9)
+- * HashAggregate (8)
   +- Exchange (7)
      +- * HashAggregate (6)
         +- * HashAggregate (5)
            +- Exchange (4)
               +- * HashAggregate (3)
                  +- * Project (2)
                     +- * Range (1)


(1) Range [codegen id : 1]
Output [1]: [id#169L]
Arguments: Range (0, 10000, step=1, splits=Some(8))

(2) Project [codegen id : 1]
Output [2]: [id#169L, (id#169L % 5) AS key#170L]
Input [1]: [id#169L]

(3) HashAggregate [codegen id : 1]
Input [2]: [id#169L, key#170L]
Keys [1]:

In [25]:
# ═══════════════════════════════════════════════════════════════════════════ #
# 5. PIPELINE FUSION — NARROW CHAINS HAVE NO INTER-OPERATOR I/O
#
#  Narrow transforms are pipelined: a row passes through the entire chain
#  (filter → withColumn → select) inside a single task with no materialisation
#  between steps.  Inserting an unnecessary wide op breaks the pipeline,
#  forcing a disk round-trip between the two halves.
# ═══════════════════════════════════════════════════════════════════════════ #
section("5. Pipeline fusion — cost of an unnecessary wide op")
print("""
  Apply the same logical transformations two ways:
    A) pure narrow chain  — filter → withColumn → withColumn → select
    B) same chain but with a repartition() injected in the middle

  Look for:  timing difference (B pays a shuffle round-trip).
             explain shows Exchange absent in A, present in B.
""")

source = spark.range(5_000_000).withColumn("dept", (F.col("id") % 10).cast("int"))

def narrow_pipeline(df):
    return (
        df
        .filter(F.col("id") > 0)
        .withColumn("doubled",  F.col("id") * 2)
        .withColumn("category", (F.col("id") % 3).cast("string"))
        .select("dept", "doubled", "category")
    )

plan_a = narrow_pipeline(source)
plan_b = narrow_pipeline(source.repartition(8))   # unnecessary shuffle injected

print("--- A: pure narrow chain ---")
plan_a.explain("formatted")

print("--- B: repartition() injected before narrow chain ---")
plan_b.explain("formatted")

timed("A — narrow chain only  ", lambda: plan_a.count())
timed("B — unnecessary shuffle", lambda: plan_b.count())


  5. Pipeline fusion — cost of an unnecessary wide op


  Apply the same logical transformations two ways:
    A) pure narrow chain  — filter → withColumn → withColumn → select
    B) same chain but with a repartition() injected in the middle

  Look for:  timing difference (B pays a shuffle round-trip).
             explain shows Exchange absent in A, present in B.

--- A: pure narrow chain ---
== Physical Plan ==
* Project (3)
+- * Filter (2)
   +- * Range (1)


(1) Range [codegen id : 1]
Output [1]: [id#208L]
Arguments: Range (0, 5000000, step=1, splits=Some(8))

(2) Filter [codegen id : 1]
Input [1]: [id#208L]
Condition : (id#208L > 0)

(3) Project [codegen id : 1]
Output [3]: [cast((id#208L % 10) as int) AS dept#209, (id#208L * 2) AS doubled#210L, cast((id#208L % 3) as string) AS category#211]
Input [1]: [id#208L]


--- B: repartition() injected before narrow chain ---
== Physical Plan ==
* Project (5)
+- Exchange (4)
   +- * Project (3)
      +- * Filter (2)
         +- * Range 

4999999

In [26]:
# ═══════════════════════════════════════════════════════════════════════════ #
# 6. FAULT RECOVERY COST — NARROW VS WIDE
#
#  Spark re-executes failed tasks using lineage.  The cost depends on where
#  in the DAG the failure occurs:
#
#  Narrow  — only the failed partition's direct ancestry needs to rerun.
#            One task reads its one parent partition and recomputes.
#
#  Wide    — if a shuffle map output is lost, ALL tasks that wrote to that
#            shuffle (the entire map stage) may need to rerun, because
#            reducers can no longer fetch their slices.
#
#  This is why checkpointing after expensive wide ops matters — it replaces
#  the full upstream lineage with a reliable file so recovery is cheap.
# ═══════════════════════════════════════════════════════════════════════════ #
section("6. Fault recovery cost — narrow vs wide lineage")
print("""
  Illustrate via lineage depth: more ancestors = more work to recover.

  Count RDD ancestors as a proxy for recovery cost.
  Narrow chain of N ops → N ancestors, all cheap (one partition each).
  Each wide op → potentially all partitions of the map stage to rerun.
""")

def lineage_depth(df) -> int:
    """Approximate lineage depth from toDebugString indentation."""
    lines = df.rdd.toDebugString().decode().splitlines()
    return max(len(line) - len(line.lstrip()) for line in lines if line.strip()) // 2

base = spark.range(10_000)

# Build progressively deeper narrow chains
n1 = base.filter(F.col("id") > 0)
n5 = n1.withColumn("a", F.col("id") + 1).withColumn("b", F.col("id") + 2) \
        .withColumn("c", F.col("id") + 3).withColumn("d", F.col("id") + 4)

# Build chains with wide ops
w1 = base.groupBy(F.col("id") % 5).agg(F.count("*"))
w3 = (
    base
    .groupBy(F.col("id") % 5).agg(F.sum("id").alias("s"))
    .groupBy("s").agg(F.count("*").alias("c"))
    .repartition(4)
)

print(f"{'DAG':<35} {'lineage depth'}")
print("-" * 50)
print(f"{'1 narrow op':<35} {lineage_depth(n1)}")
print(f"{'5 narrow ops':<35} {lineage_depth(n5)}")
print(f"{'1 wide op (groupBy)':<35} {lineage_depth(w1)}")
print(f"{'3 wide ops (groupBy×2 + repartition)':<35} {lineage_depth(w3)}")

print()
print("Full lineage for 3 wide ops:")
print(w3.rdd.toDebugString().decode())


  6. Fault recovery cost — narrow vs wide lineage


  Illustrate via lineage depth: more ancestors = more work to recover.

  Count RDD ancestors as a proxy for recovery cost.
  Narrow chain of N ops → N ancestors, all cheap (one partition each).
  Each wide op → potentially all partitions of the map stage to rerun.

DAG                                 lineage depth
--------------------------------------------------
1 narrow op                         0
5 narrow ops                        0
1 wide op (groupBy)                 2
3 wide ops (groupBy×2 + repartition) 5

Full lineage for 3 wide ops:
(4) MapPartitionsRDD[198] at javaToPython at NativeMethodAccessorImpl.java:0 []
 |  MapPartitionsRDD[197] at javaToPython at NativeMethodAccessorImpl.java:0 []
 |  SQLExecutionRDD[196] at javaToPython at NativeMethodAccessorImpl.java:0 []
 |  ShuffledRowRDD[195] at javaToPython at NativeMethodAccessorImpl.java:0 []
 +-(8) MapPartitionsRDD[194] at javaToPython at NativeMethodAccessorImpl.java:0

In [ ]:
spark.stop()